# 🚗 Car Price Prediction System

## Proje Tanımı
Bir otomotiv şirketinin araç özelliklerine bakarak 
ikinci el araba fiyatlarını tahmin eden TensorFlow tabanlı 
bir regresyon sistemi.

## 🤔 Neden Regresyon?
Tahmin etmek istediğimiz değer bir **kategori değil, sayıdır.**
Araç fiyatı 0/1 gibi sınıflarla ifade edilemez.
Bu yüzden regresyon kullanıyoruz.

## 📊 Kullanılan Veri
| Özellik | Açıklama |
|---|---|
| year | Araç model yılı |
| mileage | Kilometre |
| tax | Vergi |
| mpg | Yakıt tüketimi |
| engineSize | Motor hacmi |
| transmission | Şanzıman tipi (0/1/2/3) |
| price | Araç fiyatı (€) → Hedef değişken |

## 🛠️ Kapsanan TensorFlow Konuları
- **Veri hazırlama** → Encoding, normalizasyon
- **Train/Test ayırma** → %80 / %20
- **Sequential model** → Dense katmanlar
- **Regresyon metrikleri** → MSE, MAE
- **Model değerlendirme** → Overfitting analizi

## 📁 Dosya Yapısı
```
tensorflow_car_price_prediction.ipynb
merc.xlsx
```

In [1]:
import pandas as pd

In [2]:
dataFrame = pd.read_excel("merc.xlsx")

In [3]:
dataFrame.head()

,year,price,transmission,mileage,tax,mpg,engineSize
0,2005,5200,Automatic,63000,325,32.1,1.8
1,2017,34948,Automatic,27000,20,61.4,2.1
2,2016,49948,Automatic,6200,555,28.0,5.5
3,2016,61948,Automatic,16000,325,30.4,4.0
4,2016,73948,Automatic,4000,325,30.1,4.0


In [7]:
dataFrame['transmission'].value_counts()

transmission
Semi-Auto    6848
Automatic    4825
Manual       1444
Other           2
Name: count, dtype: int64

In [8]:
dataFrame['transmission'] = dataFrame['transmission'].map({
    'Automatic': 0,
    'Manual': 1,
    'Semi-Auto': 2,
    'Other': 3
})
# transmission sütunundaki yazıları sayılara çevirdim
# Automatic=0, Manual=1, Semi-Auto=2, Other=3
# böylece model bu sütunu okuyabilecek

In [10]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Feature ve target ayır
x = dataFrame.drop(columns=['price']).values
y = dataFrame['price'].values

# Normalizasyon
scaler = MinMaxScaler()
x = scaler.fit_transform(x)

print(f"X shape: {x.shape}")
print(f"y shape: {y.shape}")

X shape: (13119, 6)
y shape: (13119,)


In [12]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=13)

print(f"Train: {x_train.shape}")
print(f"Test:  {x_test.shape}")

Train: (10495, 6)
Test:  (2624, 6)


In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(6,)))
model.add(Dense(32, activation='relu'))
model.add(Dense(1))

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                448       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2561 (10.00 KB)
Trainable params: 2561 (10.00 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [14]:
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

In [16]:
history = model.fit(
    x_train, y_train,
    epochs=100,
    batch_size=32,
    verbose=1
)

Epoch 1/100
328/328 [==============================] - 1s 807us/step - loss: 733935616.0000 - mae: 24458.1875
Epoch 2/100
328/328 [==============================] - 0s 707us/step - loss: 619934848.0000 - mae: 21984.4199
Epoch 3/100
328/328 [==============================] - 0s 722us/step - loss: 351670496.0000 - mae: 14762.0479
Epoch 4/100
328/328 [==============================] - 0s 706us/step - loss: 162460656.0000 - mae: 8277.3955
Epoch 5/100
328/328 [==============================] - 0s 694us/step - loss: 128461304.0000 - mae: 7651.0811
Epoch 6/100
328/328 [==============================] - 0s 701us/step - loss: 124829952.0000 - mae: 7659.7749
Epoch 7/100
328/328 [==============================] - 0s 710us/step - loss: 122148672.0000 - mae: 7590.5967
Epoch 8/100
328/328 [==============================] - 0s 721us/step - loss: 119492768.0000 - mae: 7486.5610
Epoch 9/100
328/328 [==============================] - 0s 715us/step - loss: 116825928.0000 - mae: 7404.5737
Epoch 10/100
328

In [17]:
print(dataFrame['price'].describe())

count     13119.000000
mean      24698.596920
std       11842.675542
min         650.000000
25%       17450.000000
50%       22480.000000
75%       28980.000000
max      159999.000000
Name: price, dtype: float64


In [ ]:
test_loss, test_mae = model.evaluate(x_test, y_test)
print(f"Test MAE: {test_mae:.2f}€")
print(f"Test MSE: {test_loss:.2f}")

# MAE: 3953€ → Model ortalama 3953 Euro yanılıyor
# MSE: 44.339.088 → Büyük hataları daha çok cezalandırır
# MSE'nin yüksek görünmesi normal çünkü hata kareleniyor

# Overfitting kontrolü:
# Train MAE: 3814€ / Test MAE: 3953€
# İkisi birbirine çok yakın → Overfitting yok

82/82 [==============================] - 0s 670us/step - loss: 44339088.0000 - mae: 3953.3452
Test MAE: 3953.35€
Test MSE: 44339088.00


In [25]:
yeni_arac = np.array([[2019, 0, 15000, 150, 45.0, 2.0]])

# Normalize et
yeni_arac_scaled = scaler.transform(yeni_arac)

# Tahmin yap
tahmin = model.predict(yeni_arac_scaled)
print(f"Tahmini fiyat: {tahmin[0][0]:,.2f}€")

1/1 [==============================] - 0s 15ms/step
Tahmini fiyat: 26,089.94€


## 🔍 Araba Fiyatları Analizi

### Hangi Özellik Daha Etkili?
Model yılı (year) fiyat üzerinde en güçlü etkiye sahip olabilir.
Yeni model araçlar genellikle daha yüksek fiyatlıdır.
Bunu doğrulamak için ilerleyen aşamalarda feature importance analizi yapılabilir.

### Model Neyi İyi Tahmin Eder?
- **Ortalama fiyatlı araçlar** (15.000€ - 35.000€) → daha iyi tahmin
- **Çok ucuz veya çok pahalı araçlar** → daha zayıf tahmin
- Çünkü model veri dağılımının yoğun olduğu bölgede daha iyi öğrenir.

## 📋 Yönetici Özeti

- **Ortalama Tahmin Hatası:** 3.953€
- **Model İkinci El Fiyat Tahmininde Kullanılabilir**
- **Örnek Tahmin:** 2019 model, 15.000 km, 2.0 motor → 26.089€
- **Güçlü Yönler:** Ortalama fiyatlı araçlarda başarılı tahmin
- **Gelişim Alanları:** 
  - Daha fazla veri ile performans artırılabilir
  - Feature importance analizi yapılabilir
  - Epoch sayısı artırılarak MAE düşürülebilir